In [ ]:
import rosbag
from matplotlib import pyplot as plt
import numpy as np
import bagpy
from bagpy import bagreader

In [ ]:
def ReadBatteryVoltageAndFlyStatus(bag_path):
    """ Example bag file:
    path:        battery_2.bag
    version:     2.0
    duration:    9:46s (586s)
    start:       Jan 01 1970 00:05:07.17 (307.17)
    end:         Jan 01 1970 00:14:53.65 (893.65)
    size:        152.4 KB
    messages:    1469
    compression: none [1/1 chunks]
    types:       mavros_msgs/ExtendedState [ae780b1800fe17b917369d21b90058bd]
                mavros_msgs/State         [65cd0a9fff993b062b91e354554ec7e9]
                sensor_msgs/BatteryState  [4ddae7f048e32fda22cac764685e3974]
    topics:      /uav2/mavros/battery          294 msgs    : sensor_msgs/BatteryState
                /uav2/mavros/extended_state   587 msgs    : mavros_msgs/ExtendedState
                /uav2/mavros/state            588 msgs    : mavros_msgs/State
    """
    bag = rosbag.Bag(bag_path)
    battery_voltage = []
    battery_current = []
    armed = []
    in_air = []
    t0 = None
    for topic, msg, t in bag.read_messages(topics=['/uav2/mavros/battery', '/uav2/mavros/state', '/uav2/mavros/extended_state']):
        if t0 is None:
            t0 = t
        t1 = (msg.header.stamp - t0).to_sec()
        # Check if the message is of type sensor_msgs/BatteryState
        if msg._type == 'sensor_msgs/BatteryState':
            battery_voltage.append([t1, msg.voltage])
            battery_current.append([t1, -msg.current])
        # Check if the message is of type mavros_msgs/State
        if msg._type == 'mavros_msgs/State':
            armed.append([t1, msg.armed])
        if msg._type == 'mavros_msgs/ExtendedState':
            in_air.append([t1, msg.landed_state == 2])
    bag.close()
    return np.array(battery_voltage), np.array(battery_current), np.array(in_air)

def FittingRemainTime(battery_voltages, in_air, n = 5):
    # Find the time in_air first becomes True
    t_in_air = in_air[np.argmax(in_air[:,1]), 0]
    # Find the time in_air last becomes False after t_in_air
    t_landing = in_air[np.argmin(in_air[in_air[:,0] > t_in_air,1]), 0]
    print(f"Time in air starts at {t_in_air:.2f}s and ends at {t_landing:.2f}s, duration: {t_landing - t_in_air:.2f}s")

    # Now fitting the battery voltage vs remaining time, note that we ignore the first t_e_ignore and last t_e_ignore seconds
    t_e_ignore = 10
    t_s = t_in_air + t_e_ignore
    t_e = t_landing - t_e_ignore
    print(f"Fit the battery voltage vs remaining time from {t_s:.2f}s to {t_e:.2f}s")
    battery_voltages = battery_voltages[(battery_voltages[:,0] > t_s) & (battery_voltages[:,0] < t_e)]
    in_air = in_air[(in_air[:,0] > t_s) & (in_air[:,0] < t_e)]
    remaining_time = t_landing - battery_voltages[:,0]
    # Fit the battery voltage vs remaining time use n degree polynomial use np.polyfit
    n = 5
    p = np.polyfit(battery_voltages[:,1], remaining_time, n)
    print(f"Polynomial fit: {p}")
    return p
    
battery_voltages, battery_current, in_air = ReadBatteryVoltageAndFlyStatus("/home/xuhao/output/battery_2.bag")
p = FittingRemainTime(battery_voltages, in_air)

def DisplayBattery(battery_voltages, in_air, p):
    # Plot the battery voltage and fly status
    fig, ax1 = plt.subplots()
    color = 'tab:red'
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Battery Voltage (V)', color=color)
    ax1.plot(battery_voltages[:,0], battery_voltages[:,1], color=color)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid()

    # # Draw in air
    # ax2 = ax1.twinx()
    # color = 'tab:blue'
    # ax2.set_ylabel('In Air', color=color)
    # ax2.plot(in_air[:,0], in_air[:,1], color=color)
    # ax2.tick_params(axis='y', labelcolor=color)
    # ax2.grid()

    # Draw discharge rate use differential of battery voltage
    ax2 = ax1.twinx()
    color = 'tab:blue'
    ax2.set_ylabel('Discharge Rate (V/s)', color=color)
    ax2.plot(battery_voltages[1:,0], np.diff(battery_voltages[:,1]) / np.diff(battery_voltages[:,0]), color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.set_ylim(-0.05, 0.05)
    ax2.grid()
         

    # Draw the polynomial fit on ax1 on in air time
    voltage = np.linspace(15, 22, 100)
    remaining_time = np.polyval(p, voltage)
    ax1.plot(in_air[-1,0] - remaining_time, voltage, color='tab:green')
    ax1.set_xlim(in_air[0,0], in_air[-1,0])
    plt.show()
    




DisplayBattery(battery_voltages, in_air, p)
